# Assigned-intensity design audit

This executed companion reads the final 120,960-run release. The descriptive tables and initial integrity summary use the original 108,000-run R30 sample; the complete-U view uses all available R30/R60 replications. See MANUSCRIPT_MAP.md for the current submission numbering.


In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
AN = ROOT / 'analysis_outputs'
FIG = ROOT / 'figures'
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:.6g}')

## 1. Data and integrity

The combined file is derived from six archived designs and nineteen newly executed designs. Every design uses the same 144 factorial cells and 30 replications. Random-stream keys are paired across designs.

In [2]:
integrity = json.loads((AN/'integrity_summary.json').read_text(encoding='utf-8'))
checks = pd.read_csv(AN/'integrity_checks.csv')
integrity, checks.head()

({'designs': 25,
  'runs': 108000,
  'all_design_checks_pass': True,
  'no_op_numeric_max_abs_difference': 0.0,
  'no_op_array_mismatches': 0},
   design  rows  unique_cells  duplicate_keys  replications_min  \
 0   C-D1  4320           144               0                30   
 1   C-D5  4320           144               0                30   
 2  C-D10  4320           144               0                30   
 3  C-D50  4320           144               0                30   
 4    E-S  4320           144               0                30   
 
    replications_max  seed_mismatches  direct_event_mismatches  \
 0                30                0                        0   
 1                30                0                        0   
 2                30                0                        0   
 3                30                0                        0   
 4                30                0                        0   
 
    time_array_length_errors  pass  
 0               

The no-op control must reproduce the C-D50 benchmark exactly. The acceptance criterion is a maximum absolute difference of zero for all comparable numeric fields and zero time-array mismatches.

In [3]:
assert integrity['all_design_checks_pass']
assert integrity['runs'] == 108000
assert integrity['no_op_numeric_max_abs_difference'] == 0
assert integrity['no_op_array_mismatches'] == 0
print('All integrity assertions passed.')

All integrity assertions passed.


## 2. Systematic and Monte Carlo variance

For run-level outcome $y_{cr}$, systematic design sum of squares is $R\sum_c(\bar y_c-\bar y)^2$ and Monte Carlo sum of squares is $\sum_c\sum_r(y_{cr}-\bar y_c)^2$. Main and Shapley allocations use only the systematic denominator.

In [4]:
v = pd.read_csv(AN/'variance_components.csv')
m = pd.read_csv(AN/'factor_metrics.csv')
key = v[(v.outcome=='mean_total_shift') & v.design.isin(['C-D50','E-S','E-SD','CS-0','CS-45'])].copy()
key['mc_fraction_total'] = key.ss_mc/(key.ss_sys+key.ss_mc)
key[['design','ss_sys','ss_mc','mc_fraction_total']]

# Final complete-U results use R60 for the three extensions.
u = pd.read_csv(AN / "complete_u_point_estimates.csv")
u[(u.outcome == "mean_total_shift") & (u.factor == "seeding_regime") & u.design.isin(["C-D50", "CS-0", "E-SD", "PC-M01", "PC-M05"])]


,design,outcome,factor,replications,B_total_U,B_factor_U,q_proportion,rho_proportion,tau_proportion,in_sample_B_total,in_sample_B_factor
15,C-D50,mean_total_shift,seeding_regime,30,0.00336084,0.00292984,0.87176,1,1,0.00336098,0.00292987
19,CS-0,mean_total_shift,seeding_regime,30,0.000164585,8.10338e-07,0.000241112,0.000276581,0.0489716,0.000164654,8.26571e-07
47,E-SD,mean_total_shift,seeding_regime,60,6.14822e-08,9.51221e-11,2.83031e-08,3.24666e-08,1.82937e-05,6.52326e-08,9.45784e-10
87,PC-M01,mean_total_shift,seeding_regime,30,8.04019e-05,6.9528e-06,0.00206877,0.0023731,0.0239232,8.04334e-05,6.96035e-06
91,PC-M05,mean_total_shift,seeding_regime,30,2.96245e-05,2.31938e-05,0.00690119,0.00791638,0.00881462,2.96291e-05,2.31948e-05


## 3. Centered-span identification

The mean assigned dose is fixed at 0.50N while the range changes from 0 to 0.90N. Shapley values use all 16 subsets of the four factors and allocate the interaction-bundled systematic signal.

In [5]:
order=['CS-0','CS-15','CS-30','CS-45']
centered=m[(m.design.isin(order))&(m.factor=='seeding_regime')&(m.allocation=='shapley')]
centered.pivot(index='design',columns='outcome',values='p_sys').reindex(order)

outcome,conversion_rate,mean_signed_shift,mean_total_shift
design,,,
CS-0,0.00583858,0.00197449,0.00502005
CS-15,0.271161,0.281524,0.232676
CS-30,0.591095,0.608381,0.577719
CS-45,0.754228,0.762499,0.762209


In [6]:
str(Path('figures')/'figure2_contrast_and_centered_span.png')

'figures\\figure2_contrast_and_centered_span.png'

## 4. Anchor sensitivity

Dose equalization at 0.05N, 0.50N, and 0.95N sharply reduces seeding retention at every anchor, although total systematic signal increases with the common dose. Strength equalization at 0.06, 0.14, and 0.18 reduces the message contribution at every anchor, while the magnitude of the preserved seeding signal changes with strength.

In [7]:
designs=['C-D1','CS-0','D-A95','S-A06','E-S','S-A18']
m[(m.outcome=='mean_total_shift')&(m.allocation=='shapley')&(m.design.isin(designs))&(m.factor.isin(['seeding_regime','message_condition']))][['design','factor','p_sys','rho','tau']].sort_values(['design','factor'])

,design,factor,p_sys,rho,tau
3,C-D1,message_condition,0.996897,0.00557726,0.00071406
7,C-D1,seeding_regime,0.000846262,6.93196e-07,0.00071406
51,CS-0,message_condition,0.991481,0.380563,0.0489899
55,CS-0,seeding_regime,0.00502005,0.000282118,0.0489899
91,D-A95,message_condition,0.98517,1.0752,0.139297
95,D-A95,seeding_regime,0.0113902,0.00182007,0.139297
35,E-S,message_condition,0.0128179,0.0897266,0.893445
39,E-S,seeding_regime,0.986473,1.01104,0.893445
99,S-A06,message_condition,0.0149892,0.0183796,0.156503
103,S-A06,seeding_regime,0.984239,0.176701,0.156503


In [8]:
str(Path('figures')/'figure3_anchor_sensitivity.png')

'figures\\figure3_anchor_sensitivity.png'

## 5. Controls and interpretation

The negative control is exact. In the positive control, opinion reversion makes early effects decay; continuous exposure therefore retains a larger endpoint effect even at equal total dose. The seeding Shapley share rises from about 0.5% at $\mu=0$ to 8.7% at $\mu=.01$ and 78.3% at $\mu=.05$.

In [9]:
pos=m[(m.design.isin(['CS-0','PC-M01','PC-M05']))&(m.factor=='seeding_regime')&(m.allocation=='shapley')]
pos.pivot(index='design',columns='outcome',values='p_sys')

outcome,conversion_rate,mean_signed_shift,mean_total_shift
design,,,
CS-0,0.00583858,0.00197449,0.00502005
PC-M01,0.0867455,0.0870792,0.0865355
PC-M05,0.769181,0.762975,0.782838


In [10]:
str(Path('figures')/'figure5_positive_control.png')

'figures\\figure5_positive_control.png'

## 6. Exposure-response robustness

Models M1–M4 use grouped five-fold cross-validation by replication index. Cubic splines replace the former cubic polynomial. The exercise is a predictive mechanistic audit, not causal mediation.

In [11]:
e=pd.read_csv(AN/'exposure_spline_metrics.csv')
e[(e.outcome=='mean_total_shift')&(e.design.isin(['C-D5','C-D10','C-D50','CS-15','CS-30','CS-45']))][['design','model','r2_cv','rmse_cv','increment_M3_minus_M2','increment_M4_minus_M3']]

,design,model,r2_cv,rmse_cv,increment_M3_minus_M2,increment_M4_minus_M3
12,C-D5,M1 regime,0.659735,0.00451213,0.00610258,0.0034654
13,C-D5,M2 spline(exposure),0.917673,0.00221944,0.00610258,0.0034654
14,C-D5,M3 spline + regime,0.923776,0.0021356,0.00610258,0.0034654
15,C-D5,M4 regime-specific splines,0.927241,0.00208649,0.00610258,0.0034654
24,C-D10,M1 regime,0.738173,0.00775242,0.00257316,0.000784687
25,C-D10,M2 spline(exposure),0.956849,0.00314721,0.00257316,0.000784687
26,C-D10,M3 spline + regime,0.959422,0.00305193,0.00257316,0.000784687
27,C-D10,M4 regime-specific splines,0.960207,0.00302227,0.00257316,0.000784687
36,C-D50,M1 regime,0.824398,0.0243087,0.00130594,0.00020212
37,C-D50,M2 spline(exposure),0.979511,0.00830339,0.00130594,0.00020212


## Takeaways

1. The original contrast-set pattern replicates, but it does not alone identify spread.
2. Fixed-center designs show a monotonic spread effect across all three primary outcomes.
3. Equalization diagnoses strong assigned-intensity dependence across anchors, while absolute remaining signal is anchor-dependent.
4. Main-effect and Shapley rankings agree, but Shapley reallocates interactions and increases the visible message contribution in the benchmark.
5. Exact no-op reproduction and a successful structural positive control rule out mechanical deletion as the explanation for the main audit result.
6. Claims remain about this model and its design, not real-platform persuasion effects.